# Audio Retrieval — Stage 3: Contrastive Model Evaluation

**Both encoders come from the contrastively trained CLIP-style checkpoint.**

| Stage | Query Encoding | Doc Encoding | Purpose |
|-------|---------------|--------------|------------------|
| 1 ✅ | Text → TextEncoder | Text → TextEncoder | **Upper bound** |
| 2 ✅ | ASR(Audio) → TextEncoder | Text → TextEncoder | Lower bound |
| 3 ✅ | Audio → **AudioEncoder** | Text → **TextEncoder** | **Final system** |

Both encoders project into the **same 256-dim shared space** learned during contrastive training.

**Dataset:** `MedRAG/pubmed` (HuggingFace)  
**Metrics:** MRR, MAP, NDCG, Recall@1/5/10

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install -q datasets transformers torch torchaudio sentence-transformers

In [1]:
# ── Cell 2: Config — change only this cell ────────────────────────────────
CONFIG = {
    # Path to the saved contrastive model checkpoint
    # (output of the CLIP training notebook)
    "checkpoint_path": "/kaggle/input/models/premsagars2023/clap-model-384-dim-30k-train-minilm-hubert/pytorch/default/1/clip_model (2).pt",

    # How many corpus docs to use (None = full dataset)
    "corpus_size": 100_000,

    # Number of queries to evaluate (sampled from corpus titles)
    "num_queries": 200,

    # Retrieval depth
    "top_k": 10,

    # Encoding batch size — reduce if OOM
    "text_batch_size": 64,
    "audio_batch_size": 16,

    # Max token length for text encoder
    "max_length": 128,

    # Audio config (must match training)
    "sample_rate": 16000,
    "max_audio_len": 5,   # seconds

    # TTS noise SNR (must match how audio was generated)
    "snr_db": 999,

    # Output directory
    "output_dir": "/kaggle/working/results",
}

In [2]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import io
import json
import os
import random
import wave
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from datasets import load_dataset
from transformers import HubertModel, AutoModel, AutoTokenizer
from tqdm.notebook import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CONFIG['checkpoint_path']}")

os.makedirs(CONFIG["output_dir"], exist_ok=True)

Device     : cuda
Checkpoint : /kaggle/input/models/premsagars2023/clap-model-384-dim-30k-train-minilm-hubert/pytorch/default/1/clip_model (2).pt


In [3]:
# ── Cell 4: Reconstruct encoder architectures (must match training) ───────
#
# These class definitions are identical to the training notebook.
# They are needed so torch.load can restore the state dicts.

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.attn(hidden_states), dim=1)  # (B, T', 1)
        return (hidden_states * weights).sum(dim=1)                # (B, hidden_dim)


class AudioEncoder(nn.Module):
    """HuBERT + attention pooling + MLP projection → shared proj_dim space."""
    def __init__(self, audio_model_name: str, audio_hidden_dim: int, proj_dim: int):
        super().__init__()
        self.hubert  = HubertModel.from_pretrained(audio_model_name)
        self.pooling = AttentionPooling(audio_hidden_dim)
        self.proj    = nn.Sequential(
            nn.Linear(audio_hidden_dim, audio_hidden_dim),
            nn.GELU(),
            nn.LayerNorm(audio_hidden_dim),
            nn.Linear(audio_hidden_dim, proj_dim),
        )

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        # waveforms: (B, T)
        hidden = self.hubert(input_values=waveforms).last_hidden_state  # (B, T', 768)
        pooled = self.pooling(hidden)                                    # (B, 768)
        proj   = self.proj(pooled)                                       # (B, proj_dim)
        return F.normalize(proj, p=2, dim=-1)


class TextEncoder(nn.Module):
    """MiniLM + MLP projection → same shared proj_dim space as AudioEncoder."""
    def __init__(self, text_model_name: str, text_hidden_dim: int, proj_dim: int):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(text_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        self.proj      = nn.Sequential(
            nn.Linear(text_hidden_dim, text_hidden_dim),
            nn.GELU(),
            nn.LayerNorm(text_hidden_dim),
            nn.Linear(text_hidden_dim, proj_dim),
        )

    def forward(self, texts: list) -> torch.Tensor:
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=CONFIG["max_length"],
            return_tensors="pt",
        ).to(DEVICE)
        out    = self.encoder(**encoded)
        mask   = encoded["attention_mask"]
        pooled = (out.last_hidden_state * mask.unsqueeze(-1)).sum(1) / mask.sum(-1, keepdim=True)
        proj   = self.proj(pooled)
        return F.normalize(proj, p=2, dim=-1)

print("Encoder classes defined ✓")

Encoder classes defined ✓


In [4]:
# ── Cell 5: Load checkpoint ───────────────────────────────────────────────
print(f"Loading checkpoint: {CONFIG['checkpoint_path']} ...")
ckpt = torch.load(CONFIG["checkpoint_path"], map_location=DEVICE)

cfg = ckpt["cfg"]
print(f"  audio_model  : {cfg['audio_model_name']}")
print(f"  text_model   : {cfg['text_model_name']}")
print(f"  proj_dim     : {cfg['proj_dim']}")
print(f"  unfreeze_last: {cfg['unfreeze_last_n']}")
print(f"  epoch saved  : {ckpt.get('epoch', 'unknown')}")

# Instantiate encoders with saved hyperparams
audio_encoder = AudioEncoder(
    audio_model_name = cfg["audio_model_name"],
    audio_hidden_dim = cfg["audio_hidden_dim"],
    proj_dim         = cfg["proj_dim"],
).to(DEVICE)

text_encoder = TextEncoder(
    text_model_name = cfg["text_model_name"],
    text_hidden_dim = cfg["text_hidden_dim"],
    proj_dim        = cfg["proj_dim"],
).to(DEVICE)

# Restore trained weights
audio_encoder.load_state_dict(ckpt["audio_encoder"])
text_encoder.load_state_dict(ckpt["text_encoder"])

audio_encoder.eval()
text_encoder.eval()

print(f"\nAudioEncoder params : {sum(p.numel() for p in audio_encoder.parameters()):,}")
print(f"TextEncoder params  : {sum(p.numel() for p in text_encoder.parameters()):,}")
print("Checkpoint loaded ✓")

Loading checkpoint: /kaggle/input/models/premsagars2023/clap-model-384-dim-30k-train-minilm-hubert/pytorch/default/1/clip_model (2).pt ...
  audio_model  : facebook/hubert-base-ls960
  text_model   : sentence-transformers/all-MiniLM-L6-v2
  proj_dim     : 384
  unfreeze_last: 4
  epoch saved  : 10


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



AudioEncoder params : 95,259,905
TextEncoder params  : 23,009,664
Checkpoint loaded ✓


In [5]:
# ── Cell 6: Load corpus ───────────────────────────────────────────────────
print("Loading MedRAG/pubmed (streaming mode)...")
n = CONFIG["corpus_size"] or 1000

dataset = load_dataset("MedRAG/pubmed", split="train", streaming=True)
dataset = list(dataset.take(n))

print(f"Corpus size : {len(dataset):,} documents")
print(f"Columns     : {list(dataset[0].keys())}\n")
print("Sample document:")
for k, v in dataset[0].items():
    print(f"  {k}: {str(v)[:120]}")

Loading MedRAG/pubmed (streaming mode)...


Resolving data files:   0%|          | 0/1166 [00:00<?, ?it/s]

Corpus size : 100,000 documents
Columns     : ['id', 'title', 'content', 'contents', 'PMID']

Sample document:
  id: pubmed23n0001_0
  title: [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol 
  content: (--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH
  contents: [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol 
  PMID: 21


In [6]:
corpus = {}
for row in dataset:
    doc_id   = str(row["id"])
    title    = (row.get("title")    or "").strip()
    content  = (row.get("content")  or "").strip()
    contents = (row.get("contents") or f"{title} {content}").strip()
    corpus[doc_id] = {"title": title, "content": content, "contents": contents}

all_doc_ids = list(corpus.keys())

# Take first N instead of random sample
query_doc_ids = all_doc_ids[:min(200, len(all_doc_ids))]

queries = {}
qrels   = {}
for doc_id in query_doc_ids:
    qid          = f"q_{doc_id}"
    queries[qid] = corpus[doc_id]["title"]
    qrels[qid]   = {doc_id: 1}

print(f"Corpus docs : {len(corpus):,}")
print(f"Queries     : {len(queries):,}")
print(f"Qrels       : {sum(len(v) for v in qrels.values()):,} judgments")

Corpus docs : 100,000
Queries     : 200
Qrels       : 200 judgments


In [7]:
# ── Cell 8: TTS + Noise → WAV files ──────────────────────────────────────
!pip install -q piper-tts
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx.json

from piper import PiperVoice

AUDIO_DIR = "/kaggle/working/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

SNR_DB = CONFIG["snr_db"]
voice  = PiperVoice.load("en_US-lessac-low.onnx", use_cuda=(DEVICE == "cuda"))

def add_noise(audio_int16: np.ndarray, snr_db: float) -> np.ndarray:
    audio_f      = audio_int16.astype(np.float32)
    signal_power = np.mean(audio_f ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise        = np.random.normal(0, np.sqrt(noise_power), audio_f.shape)
    return np.clip(audio_f + noise, -32768, 32767).astype(np.int16)

def synthesize_and_save(text, qid):
    path = os.path.join(AUDIO_DIR, f"{qid}.wav")
    if os.path.exists(path):
        return path
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wb:
        voice.synthesize_wav(text, wb)
    buf.seek(0)
    with wave.open(buf, "rb") as wb:
        params = wb.getparams()
        frames = wb.readframes(wb.getnframes())
    audio = np.frombuffer(frames, dtype=np.int16).copy()
    with wave.open(path, "wb") as wo:
        wo.setparams(params)
        wo.writeframes(add_noise(audio, SNR_DB).tobytes())
    return path

print(f"Generating audio for {len(queries):,} queries (SNR={SNR_DB}dB)...")
query_audio = {}
for qid, text in tqdm(queries.items(), desc="TTS+Noise"):
    query_audio[qid] = synthesize_and_save(text, qid)

print(f"Done. Sample: {list(query_audio.items())[0]}")

/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Generating audio for 200 queries (SNR=999dB)...


TTS+Noise:   0%|          | 0/200 [00:00<?, ?it/s]

/tmp/ipykernel_707/1550821822.py:17: RuntimeWarning: overflow encountered in cast
  noise_power  = signal_power / (10 ** (snr_db / 10))


Done. Sample: ('q_pubmed23n0001_0', '/kaggle/working/audio/q_pubmed23n0001_0.wav')


In [8]:
# ── Cell 9: Encode corpus with TextEncoder ────────────────────────────────
#
# The contrastive TextEncoder projects MiniLM → 256-dim shared space.
# This is the space AudioEncoder also projects into, enabling cross-modal
# cosine similarity retrieval.

doc_ids   = list(corpus.keys())
doc_texts = [corpus[d]["content"] for d in doc_ids]

print(f"Encoding {len(doc_texts):,} corpus docs with TextEncoder ...")
corpus_embeddings_list = []

with torch.no_grad():
    for start in tqdm(range(0, len(doc_texts), CONFIG["text_batch_size"]), desc="Corpus"):
        batch = doc_texts[start : start + CONFIG["text_batch_size"]]
        embs  = text_encoder(batch)   # (B, proj_dim), already L2-normalised
        corpus_embeddings_list.append(embs.cpu().numpy())

corpus_embeddings = np.vstack(corpus_embeddings_list)
print(f"Corpus embeddings : {corpus_embeddings.shape}")

Encoding 100,000 corpus docs with TextEncoder ...


Corpus:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus embeddings : (100000, 384)


In [9]:
# ── Cell 10: Encode audio queries with AudioEncoder ───────────────────────
#
# WAV files → AudioEncoder → 256-dim shared space.
# No ASR step — this is true end-to-end audio retrieval.

MAX_SAMPLES = CONFIG["sample_rate"] * CONFIG["max_audio_len"]

def load_waveform(wav_path: str) -> torch.Tensor:
    """Load a WAV, resample if needed, mono, pad/trim to MAX_SAMPLES."""
    waveform, sr = torchaudio.load(wav_path)
    if sr != CONFIG["sample_rate"]:
        waveform = T.Resample(sr, CONFIG["sample_rate"])(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)   # (T,)
    if waveform.shape[0] < MAX_SAMPLES:
        waveform = F.pad(waveform, (0, MAX_SAMPLES - waveform.shape[0]))
    else:
        waveform = waveform[:MAX_SAMPLES]
    return waveform

qids       = list(query_audio.keys())
wav_paths  = [query_audio[q] for q in qids]

print(f"Encoding {len(wav_paths):,} audio queries with AudioEncoder ...")
query_embeddings_list = []

with torch.no_grad():
    for start in tqdm(range(0, len(wav_paths), CONFIG["audio_batch_size"]), desc="Audio queries"):
        batch_paths = wav_paths[start : start + CONFIG["audio_batch_size"]]
        waveforms   = torch.stack([load_waveform(p) for p in batch_paths]).to(DEVICE)
        embs        = audio_encoder(waveforms)   # (B, proj_dim), L2-normalised
        query_embeddings_list.append(embs.cpu().numpy())

query_embeddings = np.vstack(query_embeddings_list)
print(f"Query embeddings  : {query_embeddings.shape}")

Encoding 200 audio queries with AudioEncoder ...


Audio queries:   0%|          | 0/13 [00:00<?, ?it/s]

Query embeddings  : (200, 384)


In [10]:
# ── Cell 11: Retrieval ────────────────────────────────────────────────────
print("Running retrieval ...")

# cosine similarity via dot product on L2-normalised embeddings
scores      = query_embeddings @ corpus_embeddings.T        # (num_queries, num_docs)
top_k       = CONFIG["top_k"]
ranked_idxs = np.argsort(-scores, axis=1)[:, :top_k]

results = {}
for i, qid in enumerate(qids):
    results[qid] = [
        (doc_ids[idx], float(scores[i, idx]))
        for idx in ranked_idxs[i]
    ]

print(f"Retrieval done for {len(results):,} queries, top-{top_k} docs each.")

Running retrieval ...
Retrieval done for 200 queries, top-10 docs each.


In [11]:
# ── Cell 12: Evaluation ───────────────────────────────────────────────────
def compute_metrics(results, qrels, top_k=10):
    mrr_scores, map_scores, ndcg_scores = [], [], []
    recall_at = {1: [], 5: [], 10: []}

    for qid, ranked in results.items():
        relevant   = set(qrels.get(qid, {}).keys())
        if not relevant:
            continue
        ranked_ids = [doc_id for doc_id, _ in ranked]

        # MRR
        rr = 0.0
        for rank, doc_id in enumerate(ranked_ids, start=1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrr_scores.append(rr)

        # MAP
        num_seen, prec_sum = 0, 0.0
        for rank, doc_id in enumerate(ranked_ids, start=1):
            if doc_id in relevant:
                num_seen += 1
                prec_sum += num_seen / rank
        map_scores.append(prec_sum / len(relevant))

        # NDCG@k
        def dcg(ids, k):
            return sum(
                qrels[qid].get(d, 0) / np.log2(r + 1)
                for r, d in enumerate(ids[:k], start=1)
            )
        ideal_ids = sorted(relevant, key=lambda d: qrels[qid].get(d, 0), reverse=True)
        idcg      = dcg(ideal_ids, top_k)
        ndcg_scores.append(dcg(ranked_ids, top_k) / idcg if idcg > 0 else 0.0)

        # Recall@k
        for k in recall_at:
            hits = len(set(ranked_ids[:k]) & relevant)
            recall_at[k].append(hits / len(relevant))

    return {
        "MRR"          : float(np.mean(mrr_scores)),
        "MAP"          : float(np.mean(map_scores)),
        f"NDCG@{top_k}": float(np.mean(ndcg_scores)),
        "Recall@1"     : float(np.mean(recall_at[1])),
        "Recall@5"     : float(np.mean(recall_at[5])),
        "Recall@10"    : float(np.mean(recall_at[10])),
    }


metrics = compute_metrics(results, qrels, top_k=CONFIG["top_k"])

print("\n" + "="*52)
print(" STAGE 3 — CONTRASTIVE MODEL (Audio → Text)")
print(" AudioEncoder ↔ TextEncoder  |  proj_dim=256")
print("="*52)
for name, val in metrics.items():
    print(f"  {name:<15} {val:.4f}")
print("="*52)


 STAGE 3 — CONTRASTIVE MODEL (Audio → Text)
 AudioEncoder ↔ TextEncoder  |  proj_dim=256
  MRR             0.0324
  MAP             0.0324
  NDCG@10         0.0398
  Recall@1        0.0250
  Recall@5        0.0400
  Recall@10       0.0650


In [12]:
# ── Cell 13: Save results ─────────────────────────────────────────────────
ckpt_epoch = ckpt.get("epoch", "unknown")

output = {
    "stage"           : "3_contrastive",
    "checkpoint"      : CONFIG["checkpoint_path"],
    "checkpoint_epoch": ckpt_epoch,
    "audio_model"     : cfg["audio_model_name"],
    "text_model"      : cfg["text_model_name"],
    "proj_dim"        : cfg["proj_dim"],
    "corpus_size"     : len(corpus),
    "num_queries"     : len(queries),
    "snr_db"          : CONFIG["snr_db"],
    "top_k"           : CONFIG["top_k"],
    "metrics"         : metrics,
}

out_path = Path(CONFIG["output_dir"]) / f"stage3_contrastive_{CONFIG['corpus_size']}.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to: {out_path}")

Results saved to: /kaggle/working/results/stage3_contrastive_100000.json
